In [ ]:
include("main_utils.jl")
include("data_setup.jl")
include("comix_uk_time_series.jl")
include("vis_utils.jl")
include("mglm_utils.jl")

default_plot_setting()

In [ ]:
df, df_part = read_comix_uk_contact_raw();

println("# contacts: ", nrow(df))
println("# participants (rows): ", nrow(df_part))

In [ ]:
df_chunk = create_week_df(df_part)
df_part = add_date_chunks(@subset(df_part,
    Date(2021, 7, 1) .<= :date .<= Date(2021, 12, 31)))

df = innerjoin(df, @select(df_part, :part_id_d, :date,
    :chunk_number, :chunk_start, :chunk_end, :mid_date),
    on = [:part_id_d, :date])

println("# weeks: ", nrow(df_chunk))
first(df_chunk, 5)

## 1. Unweighted contact degree distribution

One degree per `(part_id_d, date)` row in the filtered participant table, stratified into `all` / `home` / `non-home`. Participant-days with no contacts in the given setting are kept as degree 0.

In [ ]:
deg_all  = contact_degrees(df, df_part; setting = :all)
deg_home = contact_degrees(df, df_part; setting = :home)
deg_non  = contact_degrees(df, df_part; setting = :nonhome)

dd_all  = DegreeDist(deg_all)
dd_home = DegreeDist(deg_home)
dd_non  = DegreeDist(deg_non)

df_dd = vcat(
    dd_to_df(dd_all,  "all"),
    dd_to_df(dd_home, "home"),
    dd_to_df(dd_non,  "non-home"),
)

p_pdf_unw  = plot_pdf_single_survey(df_dd)
p_ccdf_unw = plot_ccdf_single_survey(df_dd)
plot(p_pdf_unw, p_ccdf_unw; layout = (1, 2), size = (900, 400),
     plot_title = "Unweighted contact degree (Jul–Dec 2021)")

## 2. Weighted contact degree distribution (NA → <5 min)

Each contact contributes `duration_weight(d, β)` with `β = 0.00225 / min` and the rule from `inst/2_effective_contact_degree.md`. Missing durations are imputed as `<5min` (level 1). The weighted PDF uses a 0.5-wide histogram; the CCDF is empirical on the continuous values.

In [ ]:
β = 0.00225  # per-minute; will be inferred in future work

wdeg_all  = contact_degrees(df, df_part; setting = :all,     weighted = true, β = β)
wdeg_home = contact_degrees(df, df_part; setting = :home,    weighted = true, β = β)
wdeg_non  = contact_degrees(df, df_part; setting = :nonhome, weighted = true, β = β)

p_pdf_w  = plot_pdf_hist_single_survey(wdeg_all, wdeg_home, wdeg_non; binwidth = 0.5)
p_ccdf_w = plot_ccdf_continuous_single_survey(wdeg_all, wdeg_home, wdeg_non)
plot(p_pdf_w, p_ccdf_w; layout = (1, 2), size = (900, 400),
     plot_title = "Weighted (NA → <5min), β = $β /min")

## 3. Weighted contact degree distribution (NA imputed via 2j DM fit)

Same weight rule, but each NA-duration contact contributes its expected weight under the Dirichlet-multinomial fit on duration_multi (drop-NA-keep-n variant from 2j). The DM fit is rerun inline on the same Jul–Dec 2021 slice for `home` and `non-home`. The "all" effective degree is the per-day sum across the two settings.

In [ ]:
# Refit the DM-dropna model that 2j produces, on the same Jul–Dec 2021 slice.
inp_home = prepare_dm_inputs(df; setting = "home",     outcome = :duration_multi,
                                  K = 5, dropna_keep_n = true)
inp_non  = prepare_dm_inputs(df; setting = "non-home", outcome = :duration_multi,
                                  K = 5, dropna_keep_n = true)
fit_home = fit_mglm_dm(inp_home.X, inp_home.Y)
fit_non  = fit_mglm_dm(inp_non.X,  inp_non.Y)

# Soft NA imputation per setting; "all" is the participant-day sum across settings.
wdeg_home_imp = contact_degrees_dm_imputed(df, df_part, fit_home; setting = :home,    β = β)
wdeg_non_imp  = contact_degrees_dm_imputed(df, df_part, fit_non;  setting = :nonhome, β = β)
wdeg_all_imp  = wdeg_home_imp .+ wdeg_non_imp

p_pdf_imp  = plot_pdf_hist_single_survey(wdeg_all_imp, wdeg_home_imp, wdeg_non_imp; binwidth = 0.5)
p_ccdf_imp = plot_ccdf_continuous_single_survey(wdeg_all_imp, wdeg_home_imp, wdeg_non_imp)
plot(p_pdf_imp, p_ccdf_imp; layout = (1, 2), size = (900, 400),
     plot_title = "Weighted (NA imputed via 2j DM), β = $β /min")

## 4. Weighting comparison, stratified by setting

One figure per setting (`all` / `home` / `non-home`), each overlaying the three weighting variants on PDF + CCDF panels (PDF: 0.5-bin histogram, xlim 0–50; CCDF: continuous, log-x).

In [ ]:
plot_weighting_compare(deg_all, wdeg_all, wdeg_all_imp;
                       setting_label = "all")

In [ ]:
plot_weighting_compare(deg_home, wdeg_home, wdeg_home_imp;
                       setting_label = "home")

In [ ]:
plot_weighting_compare(deg_non, wdeg_non, wdeg_non_imp;
                       setting_label = "non-home")

## 5. β-sensitivity: CCDF overlays at β and 2β

Same per-setting layout as section 4, but CCDF only and adding the two weighted variants computed at `2β = 0.0045 / min`. Solid lines are the baseline β; dashed lines are 2β. The unweighted curve is β-free and appears once.

The DM fit itself is independent of β (the model produces proportions, which β converts to weights), so we reuse `fit_home` / `fit_non` from section 3.

In [ ]:
β2 = 1.48 * β
print_duration_weights(β)
print_duration_weights(β2)

In [ ]:

wdeg_all_2β  = contact_degrees(df, df_part; setting = :all,     weighted = true, β = β2)
wdeg_home_2β = contact_degrees(df, df_part; setting = :home,    weighted = true, β = β2)
wdeg_non_2β  = contact_degrees(df, df_part; setting = :nonhome, weighted = true, β = β2)

wdeg_home_imp_2β = contact_degrees_dm_imputed(df, df_part, fit_home; setting = :home,    β = β2)
wdeg_non_imp_2β  = contact_degrees_dm_imputed(df, df_part, fit_non;  setting = :nonhome, β = β2)
wdeg_all_imp_2β  = wdeg_home_imp_2β .+ wdeg_non_imp_2β

In [ ]:
plot_ccdf_beta_compare(deg_all, wdeg_all, wdeg_all_2β,
                       wdeg_all_imp, wdeg_all_imp_2β;
                       setting_label = "all")

In [ ]:
plot_ccdf_beta_compare(deg_home, wdeg_home, wdeg_home_2β,
                       wdeg_home_imp, wdeg_home_imp_2β;
                       setting_label = "home")

In [ ]:
plot_ccdf_beta_compare(deg_non, wdeg_non, wdeg_non_2β,
                       wdeg_non_imp, wdeg_non_imp_2β;
                       setting_label = "non-home")

## 6. Weekly mean by 5 β-patterns over the full dataset

Three-panel time series (all / home / non-home) computed on the **entire** CoMix UK dataset (2020-03-23 → 2022-03-02), unfiltered by date. This section rebuilds the participant-day, contacts, and DM-imputed pipelines on the full range independently of sections 1–5 (which are scoped to Jul–Dec 2021). Colour/linestyle match `plot_ccdf_beta_compare`.


In [ ]:
# Section 6: biweekly mean of effective degree across the full CoMix UK window.
#
# Sections 1–5 mutated `df` / `df_part` to the Jul–Dec 2021 slice, so this cell
# reloads the raw data into `*_full` names. The participant-day key is
# (:part_id_d, :date) — the same composite key used by `contact_degrees`. We
# anchor 14-day chunks to `inc2prev_week_anchor()` so chunk labels are aligned
# with the rest of the project (mirrors `add_date_chunks`, but with anchor).

df_full, df_part_full = read_comix_uk_contact_raw()

start_date_full = Date(2020, 3, 23)
end_date_full   = Date(2022, 3,  2)
df_part_full = @subset(df_part_full, start_date_full .<= :date .<= end_date_full)
df_full      = @subset(df_full,      start_date_full .<= :date .<= end_date_full)

# Anchored biweekly chunks (replicates `add_date_chunks`, with explicit anchor).
df_chunk_full = create_chunk_df(df_part_full;
    chunk_days  = 14,
    anchor_date = inc2prev_week_anchor())
df_part_full = transform(df_part_full,
    :date => ByRow(d -> assign_chunk(d, df_chunk_full)) => :chunk_number)
df_part_full = leftjoin(df_part_full, df_chunk_full, on = :chunk_number)

# Propagate chunk labels onto contact rows via the (part_id_d, date) key.
df_full = innerjoin(df_full,
    @select(df_part_full, :part_id_d, :date,
            :chunk_number, :chunk_start, :chunk_end, :mid_date),
    on = [:part_id_d, :date])

@assert all(df_part_full.chunk_start .<= df_part_full.date .<= df_part_full.chunk_end)
println("# biweekly chunks: ",   nrow(df_chunk_full))
println("# participant-days: ",  nrow(df_part_full))
println("# contact rows: ",      nrow(df_full))
println("date range: ",          extrema(df_part_full.date))

In [ ]:

# Refit the DM-dropna model on the full window (Section 3's fit was Jul–Dec 2021).
inp_home_full = prepare_dm_inputs(df_full; setting = "home",     outcome = :duration_multi,
                                            K = 5, dropna_keep_n = true)
inp_non_full  = prepare_dm_inputs(df_full; setting = "non-home", outcome = :duration_multi,
                                            K = 5, dropna_keep_n = true)
fit_home_full = fit_mglm_dm(inp_home_full.X, inp_home_full.Y)
fit_non_full  = fit_mglm_dm(inp_non_full.X,  inp_non_full.Y)

# Biweekly mean per (setting, variant). Reuses β / β2 from Sections 2 / 5.
mid_dates = sort(unique(df_part_full.mid_date))
ts = DataFrame(mid_date = Date[], setting  = String[],
               variant  = String[], mean_deg = Float64[],
               var_deg  = Float64[])

for md in mid_dates
    df_part_md = @subset(df_part_full, :mid_date .== md)
    df_md      = @subset(df_full,      :mid_date .== md)
    nrow(df_part_md) == 0 && continue

    for (setting, label) in ((:all, "all"), (:home, "home"), (:nonhome, "non-home"))
        unw  = contact_degrees(df_md, df_part_md; setting = setting)
        w_β  = contact_degrees(df_md, df_part_md; setting = setting, weighted = true, β = β)
        w_2β = contact_degrees(df_md, df_part_md; setting = setting, weighted = true, β = β2)

        if setting === :all
            imp_β  = contact_degrees_dm_imputed(df_md, df_part_md, fit_home_full; setting = :home,    β = β)  .+
                     contact_degrees_dm_imputed(df_md, df_part_md, fit_non_full;  setting = :nonhome, β = β)
            imp_2β = contact_degrees_dm_imputed(df_md, df_part_md, fit_home_full; setting = :home,    β = β2) .+
                     contact_degrees_dm_imputed(df_md, df_part_md, fit_non_full;  setting = :nonhome, β = β2)
        else
            fit_for_setting = setting === :home ? fit_home_full : fit_non_full
            imp_β  = contact_degrees_dm_imputed(df_md, df_part_md, fit_for_setting; setting = setting, β = β)
            imp_2β = contact_degrees_dm_imputed(df_md, df_part_md, fit_for_setting; setting = setting, β = β2)
        end

        push!(ts, (md, label, "unweighted",       mean(unw),    var(unw)))
        push!(ts, (md, label, "weighted, β",      mean(w_β),    var(w_β)))
        push!(ts, (md, label, "weighted, 1.5β",    mean(w_2β),   var(w_2β)))
        push!(ts, (md, label, "weighted DM, β",   mean(imp_β),  var(imp_β)))
        push!(ts, (md, label, "weighted DM, 1.5β", mean(imp_2β), var(imp_2β)))
    end
end

# Three-panel time series. Colours/linestyles match plot_ccdf_beta_compare.
_S6_STYLE = (;
    unweighted        = (color = :black,  linestyle = :solid),
    weighted_β        = (color = :orange, linestyle = :solid),
    weighted_2β       = (color = :orange, linestyle = :dash),
    weighted_DM_β     = (color = :purple, linestyle = :solid),
    weighted_DM_2β    = (color = :purple, linestyle = :dash))

_S6_VARIANTS = (
    ("unweighted",      _S6_STYLE.unweighted),
    ("weighted, β",     _S6_STYLE.weighted_β),
    ("weighted, 1.5β",    _S6_STYLE.weighted_2β),
    ("weighted DM, β",  _S6_STYLE.weighted_DM_β),
    ("weighted DM, 1.5β", _S6_STYLE.weighted_DM_2β))

function _plot_ts_panel(ts, setting_label;
                        y_col::Symbol = :mean_deg,
                        ylabel::String = "mean effective degree",
                        show_legend::Bool, kwds...)
    p = plot(; title = setting_label,
             xlabel = "mid date", ylabel = ylabel,
             legend = show_legend ? :topright : false, kwds...)
    sub = @subset(ts, :setting .== setting_label)
    for (variant, style) in _S6_VARIANTS
        df_v = sort(@subset(sub, :variant .== variant), :mid_date)
        plot!(p, df_v.mid_date, df_v[!, y_col];
              label = variant, color = style.color,
              linestyle = style.linestyle, lw = 1.5)
    end
    return p
end

p_all  = _plot_ts_panel(ts, "all";      show_legend = true)
p_home = _plot_ts_panel(ts, "home";     show_legend = false)
p_non  = _plot_ts_panel(ts, "non-home"; show_legend = false)

plot(p_all, p_home, p_non; layout = (3, 1), size = (900, 900),
     plot_title = "Biweekly mean effective degree, 5 β-patterns (full CoMix UK)")

## 7. Biweekly neighbourhood degree, 5 β-patterns

Neighbourhood degree under a configuration-model view: `mean(x) * (1 + cov(x)^2) = mean(x) + var(x) / mean(x)` — i.e. the expected degree of a randomly chosen neighbour. Computed per biweekly chunk × setting × β-pattern from the `ts` table built in Section 6 (no extra contact-degree work). Colour/linestyle match `plot_ccdf_beta_compare`.


In [ ]:
# Friendship-paradox neighbourhood degree per (mid_date, setting, variant).
# nb_deg = mean + var/mean; guard against empty / all-zero chunks.
ts = @transform(ts,
    :nb_deg = ifelse.(:mean_deg .> 0,
                      :mean_deg .+ :var_deg ./ :mean_deg,
                      0.0))

p_all_nb  = _plot_ts_panel(ts, "all";
    y_col = :nb_deg, ylabel = "neighbourhood degree", show_legend = true)
p_home_nb = _plot_ts_panel(ts, "home";
    y_col = :nb_deg, ylabel = "neighbourhood degree", show_legend = false)
p_non_nb  = _plot_ts_panel(ts, "non-home";
    y_col = :nb_deg, ylabel = "neighbourhood degree", show_legend = false)

plot(p_all_nb, p_home_nb, p_non_nb; layout = (3, 1), size = (900, 900),
     plot_title = "Biweekly neighbourhood degree, 5 β-patterns (full CoMix UK)")


In [ ]:
p_all_nb  = _plot_ts_panel(ts, "all";
    y_col = :nb_deg, ylabel = "neighbourhood degree", show_legend = true)
p_home_nb = _plot_ts_panel(ts, "home";
    y_col = :nb_deg, ylabel = "neighbourhood degree", show_legend = false)
p_non_nb  = _plot_ts_panel(ts, "non-home";
    y_col = :nb_deg, ylabel = "neighbourhood degree", show_legend = false, ylim=[0,10])

plot(p_all_nb, p_home_nb, p_non_nb; layout = (3, 1), size = (900, 900),
     plot_title = "Biweekly neighbourhood degree, 5 β-patterns (full CoMix UK)")
